# Three Methods for Robust Variable Selection: BMA, LASSO, and WALS

**Author:** Carlos Mendez | **Website:** [carlos-mendez.org](https://carlos-mendez.org/)

---

## IMPORTANT: Verify the R Runtime

This notebook requires the **R runtime**, not Python. Google Colab defaults to Python, so you must switch to R before running any cells.

**How to activate the R runtime:**

1. Go to **Runtime** > **Change runtime type** in the menu bar
2. Under **Runtime type**, select **R**
3. Click **Save**
4. The runtime will restart with R

**How to verify:** Run the cell below. If it prints the R version, you are good to go. If you get a Python error, switch the runtime first.

---

In [ ]:
# Verify R runtime is active
cat("R version:", R.version.string, "\n")
cat("If you see a Python error, go to Runtime > Change runtime type > R\n")

## Install Additional Packages

Google Colab's R runtime comes with many packages pre-installed, including `tidyverse`, `scales`, `broom`, and `ggrepel`. The cell below installs only the packages that are **not** pre-installed: `BMS`, `glmnet`, `WALS`, `corrplot`, and `patchwork`.

In [ ]:
# Only install packages NOT pre-installed in Google Colab R runtime
# Pre-installed: tidyverse, scales, broom, ggrepel (no need to reinstall)
install.packages(c(
  "BMS",         # Bayesian Model Averaging
  "glmnet",      # LASSO and Ridge regression
  "WALS",        # Weighted Average Least Squares
  "corrplot",    # correlation heatmaps
  "patchwork"    # combine ggplot panels
), repos = "https://cloud.r-project.org", quiet = TRUE)

In [ ]:
# Load libraries
library(tidyverse)
library(BMS)
library(glmnet)
library(WALS)
library(scales)
library(patchwork)
library(ggrepel)
library(corrplot)
library(broom)

# Site color palette
STEEL_BLUE  <- "#6a9bcc"
WARM_ORANGE <- "#d97757"
NEAR_BLACK  <- "#141413"
TEAL        <- "#00d4c8"
HEADING_BLUE <- "#1a3a8a"

# Custom ggplot theme
theme_site <- function(base_size = 14) {
  theme_minimal(base_size = base_size) %+replace%
    theme(
      text = element_text(color = NEAR_BLACK),
      plot.title = element_text(color = HEADING_BLUE, face = "bold", size = rel(1.1)),
      plot.subtitle = element_text(color = "gray40", size = rel(0.85)),
      legend.position = "bottom",
      panel.grid.minor = element_blank()
    )
}

---

## 1. Load the Synthetic Dataset

We use a synthetic cross-sectional dataset of 120 countries with 12 candidate regressors for $\log(\text{CO}_2)$ emissions. The dataset has a known ground truth: 7 variables have true nonzero effects and 5 are pure noise (true $\beta = 0$). The noise variables are deliberately correlated with GDP, creating realistic multicollinearity.

In [ ]:
set.seed(2021)

# Load data from GitHub
DATA_URL <- "https://raw.githubusercontent.com/cmg777/starter-academic-v501/master/content/post/r_bma_lasso_wals/synthetic-co2-cross-section.csv"
synth_data <- read.csv(DATA_URL)
cat("Dataset:", nrow(synth_data), "countries,", ncol(synth_data), "variables\n")
head(synth_data)

# Known true coefficients (our "answer key")
true_beta_lookup <- c(
  log_gdp = 1.200, industry = 0.008, fossil_fuel = 0.012,
  urban_pop = 0.010, democracy = 0.004, trade_network = 0.500,
  agriculture = 0.005, log_trade = 0, fdi = 0, corruption = 0,
  log_tourism = 0, log_credit = 0
)

## 2. Correlation Structure

The noise variables are correlated with the true predictors -- especially with GDP. This correlation is what makes variable selection difficult.

In [ ]:
cor_matrix <- synth_data |>
  select(-country, -log_co2) |>
  cor()

colnames(cor_matrix) <- c("GDP", "Industry", "Fossil fuel", "Urban pop",
                           "Democracy", "Trade net.", "Agriculture",
                           "Trade open.", "FDI", "Corruption", "Tourism", "Credit")
rownames(cor_matrix) <- colnames(cor_matrix)

corrplot(cor_matrix, method = "color", type = "lower",
         addCoef.col = NEAR_BLACK, number.cex = 0.7,
         tl.col = NEAR_BLACK, tl.cex = 0.8,
         col = colorRampPalette(c(WARM_ORANGE, "white", STEEL_BLUE))(200),
         diag = FALSE, title = "", mar = c(0, 0, 0, 0))

## 3. Naive OLS (The Problem)

With 12 correlated regressors and only 120 observations, OLS can produce misleading significance levels. Some noise variables may appear "significant" purely because they are correlated with true predictors.

In [ ]:
ols_full <- lm(log_co2 ~ log_gdp + industry + fossil_fuel + urban_pop +
                 democracy + trade_network + agriculture +
                 log_trade + fdi + corruption + log_tourism + log_credit,
               data = synth_data)
summary(ols_full)

---

# PART 1: Bayesian Model Averaging (BMA)

BMA averages across all $2^{12} = 4,096$ possible models, weighting each by its posterior probability. The **Posterior Inclusion Probability (PIP)** of a variable is the sum of posterior probabilities across all models that include it. PIP $\geq$ 0.80 indicates a robust variable.

## 4. Toy Example: BMA on 4 Variables

Before running the full model, let us enumerate all $2^4 = 16$ models with 4 variables (2 true + 2 noise) and compute PIPs by hand.

In [ ]:
toy_data <- synth_data |> select(log_co2, log_gdp, fossil_fuel, fdi, corruption)
toy_vars <- c("log_gdp", "fossil_fuel", "fdi", "corruption")

# Enumerate all 16 possible subsets
all_models <- expand.grid(
  log_gdp = c(0, 1), fossil_fuel = c(0, 1), fdi = c(0, 1), corruption = c(0, 1)
)

# Fit each model and compute BIC
model_results <- all_models |>
  mutate(model_id = row_number()) |>
  rowwise() |>
  mutate(
    included_vars = list(toy_vars[c(log_gdp, fossil_fuel, fdi, corruption) == 1]),
    n_vars = length(included_vars),
    formula_str = if (n_vars == 0) "log_co2 ~ 1"
                  else paste("log_co2 ~", paste(included_vars, collapse = " + ")),
    fit = list(lm(as.formula(formula_str), data = toy_data)),
    bic = BIC(fit)
  ) |> ungroup()

# Convert BIC to posterior probabilities
model_results <- model_results |>
  mutate(
    log_weight = -0.5 * (bic - min(bic)),
    weight = exp(log_weight),
    post_prob = weight / sum(weight)
  )

# Compute PIPs
pip_toy <- tibble(
  variable = toy_vars,
  pip = c(
    sum(model_results$post_prob[model_results$log_gdp == 1]),
    sum(model_results$post_prob[model_results$fossil_fuel == 1]),
    sum(model_results$post_prob[model_results$fdi == 1]),
    sum(model_results$post_prob[model_results$corruption == 1])
  ),
  true_effect = c("True", "True", "Noise", "Noise")
)
print(pip_toy)

## 5. BMA on All 12 Variables

Now we run BMA on the full dataset using the `BMS` package with 200,000 MCMC draws after 50,000 burn-in.

In [ ]:
set.seed(2021)

bma_data <- synth_data |>
  select(log_co2, log_gdp, industry, fossil_fuel, urban_pop,
         democracy, trade_network, agriculture,
         log_trade, fdi, corruption, log_tourism, log_credit) |>
  as.data.frame()

bma_fit <- bms(
  X.data   = bma_data,
  burn     = 50000,
  iter     = 200000,
  g        = "BRIC",
  mprior   = "uniform",
  nmodel   = 2000,
  mcmc     = "bd",
  user.int = FALSE
)

# Extract results
bma_coefs <- coef(bma_fit)
bma_df <- as.data.frame(bma_coefs) |>
  rownames_to_column("variable") |>
  as_tibble() |>
  rename(pip = PIP, post_mean = `Post Mean`, post_sd = `Post SD`) |>
  select(variable, pip, post_mean, post_sd) |>
  mutate(
    true_beta = true_beta_lookup[variable],
    robustness = case_when(
      pip >= 0.80 ~ "Robust (PIP >= 0.80)",
      pip >= 0.50 ~ "Borderline",
      TRUE        ~ "Fragile (PIP < 0.50)"
    ),
    ci_low  = post_mean - 2 * post_sd,
    ci_high = post_mean + 2 * post_sd
  )

bma_df |> arrange(desc(pip)) |> select(variable, pip, post_mean, true_beta)

In [ ]:
# PIP bar chart
ggplot(bma_df, aes(x = reorder(variable, pip), y = pip, fill = robustness)) +
  geom_col(width = 0.65) +
  geom_hline(yintercept = 0.80, linetype = "dashed", color = "gray30") +
  geom_hline(yintercept = 0.50, linetype = "dotted", color = "gray50") +
  coord_flip() +
  scale_fill_manual(values = c("Robust (PIP >= 0.80)" = STEEL_BLUE,
                                "Borderline" = TEAL,
                                "Fragile (PIP < 0.50)" = WARM_ORANGE)) +
  scale_y_continuous(limits = c(0, 1)) +
  labs(x = NULL, y = "Posterior Inclusion Probability (PIP)",
       fill = "Classification",
       title = "BMA: Posterior Inclusion Probabilities") +
  theme_site()

In [ ]:
# Posterior coefficient plot with credible intervals
ggplot(bma_df, aes(x = reorder(variable, pip), y = post_mean, color = robustness)) +
  geom_pointrange(aes(ymin = ci_low, ymax = ci_high), size = 0.5) +
  geom_hline(yintercept = 0, linetype = "solid", color = "gray50") +
  coord_flip() +
  scale_color_manual(values = c("Robust (PIP >= 0.80)" = STEEL_BLUE,
                                 "Borderline" = TEAL,
                                 "Fragile (PIP < 0.50)" = WARM_ORANGE)) +
  labs(x = NULL, y = "Posterior Mean Coefficient",
       title = "BMA: Posterior Coefficient Estimates",
       subtitle = "Error bars show approximate 95% credible intervals") +
  theme_site()

---

# PART 2: LASSO

LASSO adds an L1 penalty to the OLS objective, forcing irrelevant coefficients to exactly zero:

$$\hat{\beta}_{\text{LASSO}} = \arg\min_\beta \; \frac{1}{2n}\|y - X\beta\|^2 + \lambda \|\beta\|_1$$

The geometric insight: the L1 constraint region is a diamond whose corners force exact zeros.

## 6. L1 vs L2 Geometry

In [ ]:
# L1 diamond vs L2 circle
t_val <- 1.0
theta <- seq(0, 2 * pi, length.out = 400)
ols_b1 <- 0.3; ols_b2 <- 1.1

contour_data <- map_dfr(c(0.3, 0.6, 0.9, 1.2), function(r) {
  tibble(b1 = ols_b1 + r * 0.8 * cos(theta),
         b2 = ols_b2 + r * 0.5 * sin(theta), level = r)
})

diamond_clean <- tibble(b1 = c(t_val, 0, -t_val, 0, t_val),
                        b2 = c(0, t_val, 0, -t_val, 0))

p_l1 <- ggplot() +
  geom_path(data = contour_data, aes(x = b1, y = b2, group = level),
            color = "gray60", linetype = "dashed") +
  geom_polygon(data = diamond_clean, aes(x = b1, y = b2),
               fill = STEEL_BLUE, alpha = 0.15, color = STEEL_BLUE) +
  geom_point(aes(x = 0, y = t_val), size = 4, color = WARM_ORANGE) +
  annotate("text", x = 0.2, y = t_val + 0.1, label = "LASSO", color = WARM_ORANGE) +
  coord_equal(xlim = c(-1.5, 1.8), ylim = c(-1.5, 1.8)) +
  labs(title = "L1 (LASSO): Diamond") + theme_site()

circle <- tibble(b1 = t_val * cos(theta), b2 = t_val * sin(theta))
ols_norm <- sqrt(ols_b1^2 + ols_b2^2)

p_l2 <- ggplot() +
  geom_path(data = contour_data, aes(x = b1, y = b2, group = level),
            color = "gray60", linetype = "dashed") +
  geom_polygon(data = circle, aes(x = b1, y = b2),
               fill = TEAL, alpha = 0.15, color = TEAL) +
  geom_point(aes(x = ols_b1 * t_val / ols_norm, y = ols_b2 * t_val / ols_norm),
             size = 4, color = WARM_ORANGE) +
  annotate("text", x = 0.5, y = 1.1, label = "Ridge", color = WARM_ORANGE) +
  coord_equal(xlim = c(-1.5, 1.8), ylim = c(-1.5, 1.8)) +
  labs(title = "L2 (Ridge): Circle") + theme_site()

p_l1 + p_l2

## 7. LASSO with Cross-Validation

In [ ]:
set.seed(2021)

X <- synth_data |>
  select(log_gdp, industry, fossil_fuel, urban_pop, democracy,
         trade_network, agriculture, log_trade, fdi, corruption,
         log_tourism, log_credit) |>
  as.matrix()

y <- synth_data$log_co2

# Run LASSO with 10-fold CV
lasso_cv <- cv.glmnet(x = X, y = y, alpha = 1, nfolds = 10, standardize = TRUE)
lasso_full <- glmnet(X, y, alpha = 1, standardize = TRUE)

In [ ]:
# Regularization path
coef_path <- as.matrix(coef(lasso_full))[-1, ]
lambda_vals <- lasso_full$lambda

path_df <- as_tibble(t(coef_path)) |>
  mutate(log_lambda = log(lambda_vals)) |>
  pivot_longer(-log_lambda, names_to = "variable", values_to = "coefficient")

var_colors <- c(
  log_gdp = STEEL_BLUE, industry = STEEL_BLUE, fossil_fuel = STEEL_BLUE,
  urban_pop = STEEL_BLUE, democracy = STEEL_BLUE, trade_network = STEEL_BLUE,
  agriculture = STEEL_BLUE,
  log_trade = WARM_ORANGE, fdi = WARM_ORANGE, corruption = WARM_ORANGE,
  log_tourism = WARM_ORANGE, log_credit = WARM_ORANGE
)

ggplot(path_df, aes(x = log_lambda, y = coefficient, color = variable)) +
  geom_line(linewidth = 0.7) +
  geom_vline(xintercept = log(lasso_cv$lambda.1se), linetype = "dashed") +
  scale_color_manual(values = var_colors) +
  labs(x = expression(log(lambda)), y = "Coefficient",
       title = "LASSO Regularization Path",
       subtitle = "Steel blue = true predictors, Orange = noise") +
  theme_site() + theme(legend.position = "right")

In [ ]:
# LASSO selected variables at lambda.1se
lasso_coefs_1se <- coef(lasso_cv, s = "lambda.1se")
lasso_df <- tibble(
  variable = rownames(lasso_coefs_1se)[-1],
  lasso_coef = as.numeric(lasso_coefs_1se)[-1]
) |>
  mutate(
    selected  = lasso_coef != 0,
    true_beta = true_beta_lookup[variable],
    is_noise  = true_beta == 0
  )

cat("Variables selected at lambda.1se:\n")
lasso_df |> filter(selected) |> select(variable, lasso_coef, true_beta)

## 8. Post-LASSO

LASSO coefficients are biased (shrunk toward zero). Post-LASSO runs OLS on only the LASSO-selected variables to recover unbiased estimates.

In [ ]:
selected_vars <- lasso_df |> filter(selected) |> pull(variable)
post_lasso_formula <- as.formula(paste("log_co2 ~", paste(selected_vars, collapse = " + ")))
post_lasso_fit <- lm(post_lasso_formula, data = synth_data)

post_lasso_summary <- broom::tidy(post_lasso_fit) |>
  filter(term != "(Intercept)") |>
  rename(variable = term, post_lasso_coef = estimate) |>
  select(variable, post_lasso_coef) |>
  left_join(lasso_df |> select(variable, lasso_coef, true_beta), by = "variable")

cat("LASSO vs Post-LASSO vs True:\n")
print(post_lasso_summary)

---

# PART 3: WALS (Weighted Average Least Squares)

WALS is a frequentist model-averaging method. It splits regressors into focus ($X_1$, always included) and auxiliary ($X_2$, uncertain), then uses a semi-orthogonal transformation to solve $K$ independent averaging problems instead of $2^K$. It uses a Laplace prior -- the same prior underlying LASSO's L1 penalty.

In [ ]:
# WALS: focus = intercept, auxiliary = 12 candidate variables
X1_wals <- matrix(1, nrow = nrow(synth_data), ncol = 1)
colnames(X1_wals) <- "(Intercept)"

X2_wals <- synth_data |>
  select(log_gdp, industry, fossil_fuel, urban_pop, democracy,
         trade_network, agriculture, log_trade, fdi, corruption,
         log_tourism, log_credit) |>
  as.matrix()

y_wals <- synth_data$log_co2

wals_fit <- wals(x = X1_wals, x2 = X2_wals, y = y_wals, prior = laplace())
wals_summary <- summary(wals_fit)
aux_coefs <- wals_summary$auxCoefs

wals_df <- tibble(
  variable = rownames(aux_coefs),
  estimate = aux_coefs[, "Estimate"],
  se       = aux_coefs[, "Std. Error"],
  t_stat   = estimate / se
) |>
  mutate(
    true_beta    = true_beta_lookup[variable],
    abs_t        = abs(t_stat),
    wals_robust  = abs_t >= 2,
    true_nonzero = true_beta != 0
  )

cat("WALS results (sorted by |t|):\n")
wals_df |> arrange(desc(abs_t)) |> select(variable, estimate, t_stat, true_beta)

In [ ]:
# WALS t-statistic bar chart
wals_df <- wals_df |>
  mutate(bar_color = case_when(
    wals_robust & true_nonzero  ~ "True positive",
    wals_robust & !true_nonzero ~ "False positive",
    !wals_robust & true_nonzero ~ "False negative",
    TRUE                        ~ "True negative"
  ))

ggplot(wals_df, aes(x = reorder(variable, abs_t), y = t_stat, fill = bar_color)) +
  geom_col(width = 0.6) +
  geom_hline(yintercept = c(-2, 2), linetype = "dashed", color = "gray30") +
  coord_flip() +
  scale_fill_manual(values = c("True positive" = STEEL_BLUE,
                                "False positive" = WARM_ORANGE,
                                "True negative" = "gray70",
                                "False negative" = TEAL)) +
  labs(x = NULL, y = "t-statistic",
       title = "WALS: t-Statistics for All 12 Variables",
       subtitle = "|t| >= 2 threshold for robustness") +
  theme_site()

---

# PART 4: Grand Comparison

Which variables do all three methods agree on?

In [ ]:
# Merge all results
bma_compare <- bma_df |>
  select(variable, pip, post_mean, robustness) |>
  rename(bma_pip = pip, bma_postmean = post_mean, bma_class = robustness)

lasso_compare <- lasso_df |>
  select(variable, lasso_coef, selected) |>
  rename(lasso_selected = selected)

wals_compare <- wals_df |>
  select(variable, estimate, t_stat, wals_robust) |>
  rename(wals_estimate = estimate, wals_t = t_stat)

grand_table <- bma_compare |>
  left_join(lasso_compare, by = "variable") |>
  left_join(wals_compare, by = "variable") |>
  mutate(
    true_beta     = true_beta_lookup[variable],
    bma_robust    = bma_pip >= 0.80,
    n_methods     = bma_robust + lasso_selected + wals_robust,
    triple_robust = n_methods == 3,
    true_nonzero  = true_beta != 0
  ) |>
  arrange(desc(n_methods), desc(bma_pip))

cat("Grand Comparison Table:\n")
grand_table |>
  select(variable, true_beta, bma_pip, bma_robust, lasso_selected, wals_t, wals_robust, n_methods)

In [ ]:
# Method agreement heatmap
heatmap_df <- grand_table |>
  select(variable, true_beta, bma_robust, lasso_selected, wals_robust) |>
  pivot_longer(c(bma_robust, lasso_selected, wals_robust),
               names_to = "method", values_to = "identified") |>
  mutate(
    method = recode(method, bma_robust = "BMA",
                    lasso_selected = "LASSO", wals_robust = "WALS"),
    method = factor(method, levels = c("BMA", "LASSO", "WALS")),
    var_order = case_when(
      variable == "log_gdp" ~ 1, variable == "trade_network" ~ 2,
      variable == "fossil_fuel" ~ 3, variable == "urban_pop" ~ 4,
      variable == "industry" ~ 5, variable == "agriculture" ~ 6,
      variable == "democracy" ~ 7, variable == "log_trade" ~ 8,
      variable == "fdi" ~ 9, variable == "corruption" ~ 10,
      variable == "log_tourism" ~ 11, variable == "log_credit" ~ 12,
      TRUE ~ 13
    )
  )

ggplot(heatmap_df, aes(x = method, y = reorder(variable, -var_order), fill = identified)) +
  geom_tile(color = "white", linewidth = 1) +
  geom_text(aes(label = ifelse(identified, "Yes", "No")), size = 3.5) +
  scale_fill_manual(values = c("TRUE" = STEEL_BLUE, "FALSE" = WARM_ORANGE),
                    labels = c("Not identified", "Identified")) +
  geom_hline(yintercept = 5.5, color = NEAR_BLACK, linewidth = 1) +
  labs(x = NULL, y = NULL, fill = NULL,
       title = "Variable Identification Across Three Methods") +
  theme_site() + theme(panel.grid = element_blank())

In [ ]:
# BMA PIP vs WALS |t-statistic| scatter
grand_table_plot <- grand_table |>
  mutate(
    lasso_shape = ifelse(lasso_selected, "LASSO: Selected", "LASSO: Not selected"),
    true_status = ifelse(true_nonzero, "True predictor", "Noise")
  )

ggplot(grand_table_plot, aes(x = abs(wals_t), y = bma_pip)) +
  annotate("rect", xmin = 2, xmax = Inf, ymin = 0.80, ymax = 1,
           fill = STEEL_BLUE, alpha = 0.1) +
  geom_hline(yintercept = 0.80, linetype = "dashed", color = "gray40") +
  geom_vline(xintercept = 2, linetype = "dashed", color = "gray40") +
  geom_point(aes(color = true_status, shape = lasso_shape), size = 4) +
  geom_text_repel(aes(label = variable), size = 3, max.overlaps = 20) +
  scale_color_manual(values = c("True predictor" = STEEL_BLUE, "Noise" = WARM_ORANGE)) +
  scale_shape_manual(values = c("LASSO: Selected" = 17, "LASSO: Not selected" = 4)) +
  labs(x = "WALS |t-statistic|", y = "BMA PIP",
       title = "BMA PIP vs. WALS |t-statistic|",
       subtitle = "Upper-right = robust by both; triangles = LASSO-selected") +
  theme_site()

In [ ]:
# Method performance
results_by_method <- tibble(
  method = c("BMA", "LASSO", "WALS"),
  true_pos = c(
    sum(grand_table$bma_robust & grand_table$true_nonzero),
    sum(grand_table$lasso_selected & grand_table$true_nonzero),
    sum(grand_table$wals_robust & grand_table$true_nonzero)
  ),
  false_pos = c(
    sum(grand_table$bma_robust & !grand_table$true_nonzero),
    sum(grand_table$lasso_selected & !grand_table$true_nonzero),
    sum(grand_table$wals_robust & !grand_table$true_nonzero)
  ),
  sensitivity = true_pos / 7,
  specificity = (5 - false_pos) / 5,
  accuracy = (true_pos + 5 - false_pos) / 12
)

cat("Method Performance:\n")
print(results_by_method)

cat("\nTriple-robust variables:\n")
triple_robust_vars <- grand_table |> filter(triple_robust) |> pull(variable)
cat(paste(triple_robust_vars, collapse = ", "), "\n")

---

## Conclusion

All three methods achieved identical 83.3% accuracy, correctly identifying 5 of the 7 true predictors while producing zero false positives. The convergence across fundamentally different statistical paradigms -- Bayesian (BMA), penalized likelihood (LASSO), and frequentist model averaging (WALS) -- provides strong evidence for methodological triangulation.

**The strongest recommendation: use all three.** When they converge on the same variables, you have robust evidence.

---

*Full tutorial with detailed explanations: [carlos-mendez.org/post/r_bma_lasso_wals/](https://carlos-mendez.org/post/r_bma_lasso_wals/)*